# Harga Rumah di Jakarta: Faktor Apa yang Paling Berpengaruh?

**Materi: Exploratory Data Analysis (EDA).**

Data asli dari Kaggle: 10.000 listing rumah di Jakarta — harga, wilayah, luas tanah, luas bangunan, jumlah kamar.
Sumber: `abiyyurasyiq/jakarta-house-price-dataset`.

Pertanyaan yang mau dijawab: kalau mau menebak harga sebuah rumah, faktor apa yang paling menentukan?

In [ ]:
import zipfile, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

print("Siap! Peralatan sudah dimuat.")

## 1. Ambil data dari Kaggle

In [ ]:
url = "https://www.kaggle.com/api/v1/datasets/download/abiyyurasyiq/jakarta-house-price-dataset"
zip_path = "jakarta_house.zip"
csv_path = "jakarta_house.csv"

if not os.path.exists(csv_path):
    print("Mengunduh dari Kaggle...")
    os.system(f"curl -sL -m 120 -o {zip_path} {url}")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(".")
    print("Selesai.")
else:
    print("File sudah ada, unduhan dilewati.")

df = pd.read_csv(csv_path)
print("Jumlah baris:", len(df), "| Jumlah kolom:", len(df.columns))

## 2. Intip datanya

In [ ]:
df.head()

In [ ]:
print("Tipe data:")
print(df.dtypes)
print("\nData kosong per kolom:")
print(df.isna().sum())
print("\nRingkasan angka:")
df.describe()

## 3. Bersih-bersih

In [ ]:
sebelum = len(df)
df = df.drop(columns=["index"])   # kolom nomor urut, tidak dipakai
df = df.dropna().reset_index(drop=True)
print(f"Baris sebelum: {sebelum} -> sesudah dibersihkan: {len(df)}")
df["price_m"] = df["price"] / 1e9   # bikin kolom harga dalam miliar biar gampang dibaca

## 4. Peta korelasi — faktor mana yang bergerak bersama harga?

In [ ]:
kolom = ["price", "bed_rooms", "bath_rooms", "carport", "land_area", "building_area"]
kor = df[kolom].corr()
plt.figure(figsize=(7, 5))
plt.imshow(kor, cmap="Greens", vmin=-1, vmax=1)
plt.xticks(range(len(kolom)), kolom, rotation=25, ha="right")
plt.yticks(range(len(kolom)), kolom)
plt.colorbar(label="korelasi")
plt.title("Faktor mana yang bergerak bersama harga?")
for i in range(len(kolom)):
    for j in range(len(kolom)):
        plt.text(j, i, f"{kor.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.tight_layout(); plt.show()
print("Korelasi dengan harga, dari tertinggi:")
print(kor["price"].sort_values(ascending=False).round(2))

## 5. Sebaran harga — kebanyakan rumah di harga berapa?

Harga rumah itu miring ke kanan: kebanyakan menumpuk di harga rendah-menengah,
lalu ada sedikit yang sangat mahal.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["price_m"], bins=60, color="green", edgecolor="white")
plt.xlim(0, 60)
plt.xlabel("Harga (miliar rupiah)")
plt.ylabel("Jumlah rumah")
plt.title("Sebaran harga rumah di Jakarta (dipotong di 60 M)")
plt.tight_layout(); plt.show()
print(f"Median harga: Rp {df['price_m'].median():.2f} miliar")
print(f"Rata-rata  : Rp {df['price_m'].mean():.2f} miliar  (ketarik ke atas oleh rumah super mahal)")

## 6. Beda wilayah, beda harga

In [ ]:
med = df.groupby("city")["price_m"].median().sort_values()
plt.figure(figsize=(7, 4))
plt.barh(med.index, med.values, color="green")
plt.xlabel("Median harga (miliar rupiah)")
plt.title("Median harga rumah per wilayah Jakarta")
for i, v in enumerate(med.values):
    plt.text(v + 0.15, i, f"{v:.2f} M", va="center")
plt.tight_layout(); plt.show()

## 7. Luas bangunan vs harga

In [ ]:
sampel = df.sample(2000, random_state=7)
plt.figure(figsize=(7, 5))
plt.scatter(sampel["building_area"], sampel["price_m"], alpha=0.25, s=12, color="green")
plt.xlim(0, 1200); plt.ylim(0, 60)
plt.xlabel("Luas bangunan (m2)")
plt.ylabel("Harga (miliar rupiah)")
plt.title("Makin luas bangunan, makin mahal (2.000 sampel)")
plt.tight_layout(); plt.show()

## 8. Model sederhana: menebak harga dari luas bangunan

Regresi linear: seberapa besar pengaruh tiap 1 m2 luas bangunan terhadap harga?

In [ ]:
model1 = smf.ols("price ~ building_area", data=df).fit()
coef_jt = model1.params["building_area"] / 1e6
print(f"Tiap +1 m2 luas bangunan  ->  +Rp {coef_jt:.1f} juta")
print(f"Skor kecocokan model (R2) : {model1.rsquared:.0%}")
print("\nArtinya: luas bangunan saja sudah menjelaskan "
      f"{model1.rsquared:.0%} naik-turun harga. Sisanya dijelaskan faktor lain.")

## 9. Model lebih lengkap: tambah luas tanah dan jumlah kamar

In [ ]:
model2 = smf.ols("price ~ building_area + land_area + bed_rooms", data=df).fit()
print(f"Skor kecocokan model (R2) : {model2.rsquared:.0%}")
print("\nPengaruh masing-masing (yang lain dianggap tetap):")
print(f"  +1 m2 luas bangunan -> +Rp {model2.params['building_area']/1e6:.1f} juta")
print(f"  +1 m2 luas tanah     -> +Rp {model2.params['land_area']/1e6:.1f} juta")
print(f"  +1 kamar tidur       ->  Rp {model2.params['bed_rooms']/1e6:+.1f} juta")
print("\nCatatan: angka kamar tidur kecil dan goyang — wajar, karena jumlah kamar")
print("biasanya ikut besar kalau bangunannya besar (saling berkaitan).")

## Kesimpulan

- Median harga rumah di Jakarta pada data ini: **Rp 4,5 miliar**.
- **Luas tanah (korelasi 0,70)** dan **luas bangunan (0,63)** adalah penentu utama harga — jauh di atas jumlah kamar.
- Beda wilayah beda harga: median Jakarta Selatan Rp 8,8 M, Jakarta Timur Rp 2,45 M.
- Model sederhana (luas bangunan saja) menjelaskan 39% variasi harga; model lengkap 52%.
- Pelajaran EDA-nya: mulai dari intip data, bersihkan, gambar, baru model.
  Model tidak menjawab semuanya — tapi bikin tebakan jadi jauh lebih terukur.